##### Modulated VBM

In [2]:
#import statements
import pandas as pd
import matplotlib.pyplot as plt
import subprocess
import numpy as np
import nilearn.image
import nilearn.plotting
import tempfile
import copy
import shutil
from torch.utils.data import random_split
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
from pathlib import Path
import ants
import pydicom
import nibabel as nib
import os
from glob import glob
from tqdm import tqdm
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from torchinfo import summary
import nilearn

os.environ.setdefault("FSLDIR", "/Users/william.wakefield/fsl")
fsl_bin = f"{os.environ['FSLDIR']}/share/fsl/bin"
if fsl_bin not in os.environ.get("PATH", "").split(os.pathsep):
    os.environ["PATH"] = fsl_bin + os.pathsep + os.environ.get("PATH", "")
os.environ.setdefault("FSLOUTPUTTYPE", "NIFTI_GZ")

'NIFTI_GZ'

In [13]:
vbm_root = Path("model_data/adni/t1_long_data/t1_long_modulated_vbm")
raw_dir = vbm_root / "t1_long_raw"
strip_dir = vbm_root / "t1_long_skull_strip"

raw_files = sorted(raw_dir.glob("*.nii"))
print(f"Found {len(raw_files)} raw native T1 images to skull-strip")

Found 802 raw native T1 images to skull-strip


In [14]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def _run_bet(raw_path, out_nii):
    result = subprocess.run(
        ["bet", str(raw_path), str(out_nii), "-R"],
        capture_output=True, text=True,
    )
    return raw_path.stem, result.returncode, result.stderr.strip()

failed = []
skipped = 0
tasks = []

for raw_path in raw_files:
    out_nii = strip_dir / f"{raw_path.stem}.nii.gz"
    if out_nii.exists():
        skipped += 1
        continue
    tasks.append((raw_path, out_nii))

max_workers = max(1, (os.cpu_count() or 4) - 2)
print(f"Running {len(tasks)} BETs in parallel with {max_workers} workers "
      f"({skipped} skipped, already existed)")

with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = [ex.submit(_run_bet, raw, out) for raw, out in tasks]
    for fut in tqdm(as_completed(futures), total=len(tasks), desc="BET native T1"):
        stem, rc, err = fut.result()
        if rc != 0:
            failed.append((stem, err))

stripped = list(strip_dir.glob("*.nii.gz"))
print(f"\nNative skull stripping complete: {len(stripped)} images")
print(f"  ({skipped} skipped, already existed)")
if failed:
    print(f"{len(failed)} failures:")
    for name, err in failed[:10]:
        print(f"  {name}: {err}")

Running 802 BETs in parallel with 14 workers (0 skipped, already existed)


BET native T1: 100%|██████████| 802/802 [12:59<00:00,  1.03it/s]


Native skull stripping complete: 802 images
  (0 skipped, already existed)


In [17]:
vbm_root = Path("model_data/adni/t1_long_data/t1_long_modulated_vbm")
strip_dir = vbm_root / "t1_long_skull_strip"
seg_dir = vbm_root / "t1_long_seg_native"

stripped_files = sorted(strip_dir.glob("*.nii.gz"))
print(f"Found {len(stripped_files)} native skull-stripped T1 images to segment")

Found 802 native skull-stripped T1 images to segment


In [18]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def _run_fast(strip_path, out_base):
    result = subprocess.run(
        [
            "fast",
            "-t", "1",
            "-n", "3",
            "-B",
            "-b",
            "-o", str(out_base),
            str(strip_path),
        ],
        capture_output=True, text=True,
    )
    stem = strip_path.name.replace(".nii.gz", "")
    return stem, result.returncode, result.stderr.strip()

failed = []
skipped = 0
tasks = []

for strip_path in stripped_files:
    stem = strip_path.name.replace(".nii.gz", "")
    subject_dir = seg_dir / stem
    subject_dir.mkdir(parents=True, exist_ok=True)
    out_base = subject_dir / stem

    pve_csf = subject_dir / f"{stem}_pve_0.nii.gz"
    pve_gm = subject_dir / f"{stem}_pve_1.nii.gz"
    pve_wm = subject_dir / f"{stem}_pve_2.nii.gz"

    if pve_csf.exists() and pve_gm.exists() and pve_wm.exists():
        skipped += 1
        continue

    tasks.append((strip_path, out_base))

max_workers = max(1, (os.cpu_count() or 4) - 2)
print(f"Running {len(tasks)} FASTs in parallel with {max_workers} workers "
      f"({skipped} skipped, already existed)")

with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = [ex.submit(_run_fast, sp, ob) for sp, ob in tasks]
    for fut in tqdm(as_completed(futures), total=len(tasks), desc="FAST native T1"):
        stem, rc, err = fut.result()
        if rc != 0:
            failed.append((stem, err))

csf_maps = list(seg_dir.glob("*/*_pve_0.nii.gz"))
gm_maps = list(seg_dir.glob("*/*_pve_1.nii.gz"))
wm_maps = list(seg_dir.glob("*/*_pve_2.nii.gz"))
print(f"\nNative FAST segmentation complete:")
print(f"  {len(csf_maps)} CSF maps (pve_0)")
print(f"  {len(gm_maps)} GM maps (pve_1)")
print(f"  {len(wm_maps)} WM maps (pve_2)")
print(f"  ({skipped} skipped, already existed)")
if failed:
    print(f"{len(failed)} failures:")
    for name, err in failed[:10]:
        print(f"  {name}: {err}")

Running 802 FASTs in parallel with 14 workers (0 skipped, already existed)


FAST native T1: 100%|██████████| 802/802 [2:24:12<00:00, 10.79s/it]  


Native FAST segmentation complete:
  802 CSF maps (pve_0)
  802 GM maps (pve_1)
  802 WM maps (pve_2)
  (0 skipped, already existed)


In [7]:
vbm_root = Path("model_data/adni/t1_long_data/t1_long_modulated_vbm")
warps_dir = vbm_root / "t1_long_vbm_warps"
jac_dir = vbm_root / "t1_long_jacobian"

mni_t1_path = Path("model_data/mni_t1.nii")
mni_t1 = ants.image_read(str(mni_t1_path))

warp_files = sorted(warps_dir.glob("*_1Warp.nii.gz"))
print(f"Found {len(warp_files)} subject warps to compute Jacobians for")

Found 802 subject warps to compute Jacobians for


In [9]:
vbm_root = Path("model_data/adni/t1_long_data/t1_long_modulated_vbm")
pve_mni_dir = vbm_root / "t1_long_pve_mni"
jac_dir = vbm_root / "t1_long_jacobian"
mod_dir = vbm_root / "t1_long_pve_mni_modulated"

subject_dirs = sorted([p for p in pve_mni_dir.iterdir() if p.is_dir()])
print(f"Found {len(subject_dirs)} subjects to modulate")

Found 802 subjects to modulate


In [10]:
failed = []
skipped = 0

for subj_dir in tqdm(subject_dirs, desc="Modulate PVE * Jacobian"):
    stem = subj_dir.name
    out_subj_dir = mod_dir / stem
    out_subj_dir.mkdir(parents=True, exist_ok=True)

    jac_path = jac_dir / f"{stem}_jacobian.nii.gz"
    if not jac_path.exists():
        failed.append((stem, "missing jacobian"))
        continue

    all_done = all(
        (out_subj_dir / f"{stem}_pve_{i}_mod.nii.gz").exists() for i in (0, 1, 2)
    )
    if all_done:
        skipped += 1
        continue

    try:
        jac_img = ants.image_read(str(jac_path))
        jac_arr = jac_img.numpy()

        for tissue_idx in (0, 1, 2):
            in_pve = subj_dir / f"{stem}_pve_{tissue_idx}_mni.nii.gz"
            out_pve = out_subj_dir / f"{stem}_pve_{tissue_idx}_mod.nii.gz"

            if out_pve.exists():
                continue
            if not in_pve.exists():
                failed.append((stem, f"missing pve_{tissue_idx}_mni"))
                continue

            pve_img = ants.image_read(str(in_pve))
            mod_arr = pve_img.numpy() * jac_arr
            mod_img = ants.from_numpy(
                mod_arr,
                origin=pve_img.origin,
                spacing=pve_img.spacing,
                direction=pve_img.direction,
            )
            ants.image_write(mod_img, str(out_pve))

    except Exception as e:
        failed.append((stem, str(e)))

csf_mod = list(mod_dir.glob("*/*_pve_0_mod.nii.gz"))
gm_mod = list(mod_dir.glob("*/*_pve_1_mod.nii.gz"))
wm_mod = list(mod_dir.glob("*/*_pve_2_mod.nii.gz"))
print(f"\nModulation complete:")
print(f"  {len(csf_mod)} CSF, {len(gm_mod)} GM, {len(wm_mod)} WM modulated PVEs")
print(f"  ({skipped} skipped, already existed)")
if failed:
    print(f"{len(failed)} failures:")
    for name, err in failed[:10]:
        print(f"  {name}: {err}")

Modulate PVE * Jacobian: 100%|██████████| 802/802 [03:10<00:00,  4.22it/s]


Modulation complete:
  802 CSF, 802 GM, 802 WM modulated PVEs
  (0 skipped, already existed)


In [11]:
vbm_root = Path("model_data/adni/t1_long_data/t1_long_modulated_vbm")
mod_dir = vbm_root / "t1_long_pve_mni_modulated"
smooth_dir = vbm_root / "t1_long_pve_mni_smoothed"

fwhm_mm = 2.5
sigma_mm = fwhm_mm / (2.0 * np.sqrt(2.0 * np.log(2.0)))
print(f"Smoothing at {fwhm_mm} mm FWHM (sigma = {sigma_mm:.4f} mm)")

subject_dirs = sorted([p for p in mod_dir.iterdir() if p.is_dir()])
print(f"Found {len(subject_dirs)} subjects to smooth")

Smoothing at 2.5 mm FWHM (sigma = 1.0617 mm)
Found 802 subjects to smooth


In [12]:
failed = []
skipped = 0

for subj_dir in tqdm(subject_dirs, desc="Smooth modulated PVEs"):
    stem = subj_dir.name
    out_subj_dir = smooth_dir / stem
    out_subj_dir.mkdir(parents=True, exist_ok=True)

    all_done = all(
        (out_subj_dir / f"{stem}_pve_{i}_mod_s2p5.nii.gz").exists() for i in (0, 1, 2)
    )
    if all_done:
        skipped += 1
        continue

    try:
        for tissue_idx in (0, 1, 2):
            in_pve = subj_dir / f"{stem}_pve_{tissue_idx}_mod.nii.gz"
            out_pve = out_subj_dir / f"{stem}_pve_{tissue_idx}_mod_s2p5.nii.gz"

            if out_pve.exists():
                continue
            if not in_pve.exists():
                failed.append((stem, f"missing pve_{tissue_idx}_mod"))
                continue

            img = ants.image_read(str(in_pve))
            smoothed = ants.smooth_image(
                img,
                sigma=sigma_mm,
                sigma_in_physical_coordinates=True,
            )
            ants.image_write(smoothed, str(out_pve))

    except Exception as e:
        failed.append((stem, str(e)))

csf_s = list(smooth_dir.glob("*/*_pve_0_mod_s2p5.nii.gz"))
gm_s = list(smooth_dir.glob("*/*_pve_1_mod_s2p5.nii.gz"))
wm_s = list(smooth_dir.glob("*/*_pve_2_mod_s2p5.nii.gz"))
print(f"\nSmoothing complete:")
print(f"  {len(csf_s)} CSF, {len(gm_s)} GM, {len(wm_s)} WM smoothed maps")
print(f"  ({skipped} skipped, already existed)")
if failed:
    print(f"{len(failed)} failures:")
    for name, err in failed[:10]:
        print(f"  {name}: {err}")

Smooth modulated PVEs: 100%|██████████| 802/802 [03:38<00:00,  3.67it/s]


Smoothing complete:
  802 CSF, 802 GM, 802 WM smoothed maps
  (0 skipped, already existed)


In [8]:
failed = []
skipped = 0

for warp_path in tqdm(warp_files, desc="Jacobian (warp * affine)"):
    stem = warp_path.name.replace("_1Warp.nii.gz", "")
    affine_path = warps_dir / f"{stem}_0GenericAffine.mat"
    out_jac = jac_dir / f"{stem}_jacobian.nii.gz"

    if out_jac.exists():
        skipped += 1
        continue

    if not affine_path.exists():
        failed.append((stem, "missing affine"))
        continue

    try:
        jac_warp = ants.create_jacobian_determinant_image(
            domain_image=mni_t1,
            tx=str(warp_path),
            do_log=False,
            geom=False,
        )

        affine_tx = ants.read_transform(str(affine_path))
        params = np.asarray(affine_tx.parameters)
        A = params[:9].reshape(3, 3)
        det_affine = float(abs(np.linalg.det(A)))

        jac_total = jac_warp * det_affine
        ants.image_write(jac_total, str(out_jac))

    except Exception as e:
        failed.append((stem, str(e)))

jac_maps = list(jac_dir.glob("*_jacobian.nii.gz"))
print(f"\nJacobian computation complete: {len(jac_maps)} maps")
print(f"  ({skipped} skipped, already existed)")
if failed:
    print(f"{len(failed)} failures:")
    for name, err in failed[:10]:
        print(f"  {name}: {err}")

Jacobian (warp * affine): 100%|██████████| 802/802 [04:37<00:00,  2.88it/s]


Jacobian computation complete: 802 maps
  (0 skipped, already existed)


In [5]:
vbm_root = Path("model_data/adni/t1_long_data/t1_long_modulated_vbm")
seg_dir = vbm_root / "t1_long_seg_native"
warps_dir = vbm_root / "t1_long_vbm_warps"
pve_mni_dir = vbm_root / "t1_long_pve_mni"

mni_t1_path = Path("model_data/mni_t1.nii")
mni_t1 = ants.image_read(str(mni_t1_path))

subject_dirs = sorted([p for p in seg_dir.iterdir() if p.is_dir()])
print(f"Found {len(subject_dirs)} segmented subjects to warp into MNI")

Found 802 segmented subjects to warp into MNI


In [6]:
failed = []
skipped = 0

for subj_dir in tqdm(subject_dirs, desc="Warp PVEs -> MNI"):
    stem = subj_dir.name
    out_subj_dir = pve_mni_dir / stem
    out_subj_dir.mkdir(parents=True, exist_ok=True)

    affine = warps_dir / f"{stem}_0GenericAffine.mat"
    warp = warps_dir / f"{stem}_1Warp.nii.gz"

    if not affine.exists() or not warp.exists():
        failed.append((stem, "missing transforms"))
        continue

    transformlist = [str(warp), str(affine)]

    all_done = True
    for tissue_idx in (0, 1, 2):
        out_pve = out_subj_dir / f"{stem}_pve_{tissue_idx}_mni.nii.gz"
        if not out_pve.exists():
            all_done = False
            break
    if all_done:
        skipped += 1
        continue

    try:
        for tissue_idx in (0, 1, 2):
            in_pve = subj_dir / f"{stem}_pve_{tissue_idx}.nii.gz"
            out_pve = out_subj_dir / f"{stem}_pve_{tissue_idx}_mni.nii.gz"

            if out_pve.exists():
                continue
            if not in_pve.exists():
                failed.append((stem, f"missing pve_{tissue_idx}"))
                continue

            moving = ants.image_read(str(in_pve))
            warped = ants.apply_transforms(
                fixed=mni_t1,
                moving=moving,
                transformlist=transformlist,
                interpolator="linear",
            )
            ants.image_write(warped, str(out_pve))

    except Exception as e:
        failed.append((stem, str(e)))

csf_maps = list(pve_mni_dir.glob("*/*_pve_0_mni.nii.gz"))
gm_maps = list(pve_mni_dir.glob("*/*_pve_1_mni.nii.gz"))
wm_maps = list(pve_mni_dir.glob("*/*_pve_2_mni.nii.gz"))
print(f"\nWarp PVEs to MNI complete:")
print(f"  {len(csf_maps)} CSF, {len(gm_maps)} GM, {len(wm_maps)} WM in MNI space")
print(f"  ({skipped} skipped, already existed)")
if failed:
    print(f"{len(failed)} failures:")
    for name, err in failed[:10]:
        print(f"  {name}: {err}")

Warp PVEs -> MNI: 100%|██████████| 802/802 [08:46<00:00,  1.52it/s]


Warp PVEs to MNI complete:
  802 CSF, 802 GM, 802 WM in MNI space
  (0 skipped, already existed)


In [3]:
vbm_root = Path("model_data/adni/t1_long_data/t1_long_modulated_vbm")
raw_dir = vbm_root / "t1_long_raw"
warps_dir = vbm_root / "t1_long_vbm_warps"
warps_dir.mkdir(parents=True, exist_ok=True)

mni_t1_path = Path("model_data/mni_t1.nii")
mni_t1 = ants.image_read(str(mni_t1_path))

raw_files = sorted(raw_dir.glob("*.nii"))
print(f"Found {len(raw_files)} raw native T1 images to register")

Found 802 raw native T1 images to register


In [4]:
failed = []
skipped = 0

for raw_path in tqdm(raw_files, desc="SyN raw T1 -> MNI"):
    stem = raw_path.stem
    out_warped = warps_dir / f"{stem}_warpedT1.nii.gz"
    out_affine = warps_dir / f"{stem}_0GenericAffine.mat"
    out_warp = warps_dir / f"{stem}_1Warp.nii.gz"
    out_inv_warp = warps_dir / f"{stem}_1InverseWarp.nii.gz"

    if (
        out_warped.exists()
        and out_affine.exists()
        and out_warp.exists()
        and out_inv_warp.exists()
    ):
        skipped += 1
        continue

    try:
        moving = ants.image_read(str(raw_path))
        reg = ants.registration(fixed=mni_t1, moving=moving, type_of_transform="SyN")

        ants.image_write(reg["warpedmovout"], str(out_warped))

        for tx_path in reg["fwdtransforms"]:
            tx = Path(tx_path)
            if "GenericAffine" in tx.name:
                shutil.copy(tx, out_affine)
            elif "Warp" in tx.name and "Inverse" not in tx.name:
                shutil.copy(tx, out_warp)

        for tx_path in reg["invtransforms"]:
            tx = Path(tx_path)
            if "InverseWarp" in tx.name:
                shutil.copy(tx, out_inv_warp)

    except Exception as e:
        failed.append((stem, str(e)))

warped = list(warps_dir.glob("*_warpedT1.nii.gz"))
affines = list(warps_dir.glob("*_0GenericAffine.mat"))
fwd_warps = list(warps_dir.glob("*_1Warp.nii.gz"))
inv_warps = list(warps_dir.glob("*_1InverseWarp.nii.gz"))
print(f"\nVBM SyN registration complete:")
print(f"  {len(warped)} warped T1s")
print(f"  {len(affines)} affines, {len(fwd_warps)} forward warps, {len(inv_warps)} inverse warps")
print(f"  ({skipped} skipped, already existed)")
if failed:
    print(f"{len(failed)} failures:")
    for name, err in failed[:10]:
        print(f"  {name}: {err}")

SyN raw T1 -> MNI: 100%|██████████| 802/802 [46:16<00:00,  3.46s/it]


VBM SyN registration complete:
  802 warped T1s
  802 affines, 802 forward warps, 802 inverse warps
  (0 skipped, already existed)


In [31]:
import re
from concurrent.futures import ThreadPoolExecutor, as_completed

t1_strip_dir = Path("model_data/adni/t1_long_data/t1_long_modulated_vbm/t1_long_skull_strip")
t1_warps_dir = Path("model_data/adni/t1_long_data/t1_long_modulated_vbm/t1_long_vbm_warps")

dti_vbm_root = Path("model_data/adni/dti_long_data/dti_long_modulated_vbm")
dtifit_dir = dti_vbm_root / "dti_long_ditfit"
md_t1_dir = dti_vbm_root / "dti_long_reg_t1_native"

subj_pat = re.compile(r"^I\d+_(\d{3}_S_\d+)$")

md_maps = sorted(dtifit_dir.glob("*/*_MD.nii.gz"))
print(f"Found {len(md_maps)} DTI MD maps")

Found 801 DTI MD maps


In [32]:
t1_by_subj = {}
for f in sorted(t1_strip_dir.glob("*.nii.gz")):
    stem = f.name.replace(".nii.gz", "")
    m = subj_pat.match(stem)
    if m:
        t1_by_subj.setdefault(m.group(1), []).append(stem)

dti_by_subj = {}
for md_path in md_maps:
    dti_stem = md_path.name.replace("_MD.nii.gz", "")
    m = subj_pat.match(dti_stem)
    if m:
        dti_by_subj.setdefault(m.group(1), []).append((dti_stem, md_path))

dti_to_t1 = {}
for subj, dti_list in dti_by_subj.items():
    t1_list = sorted(t1_by_subj.get(subj, []))
    dti_list_sorted = sorted(dti_list, key=lambda x: x[0])
    for i, (dti_stem, _) in enumerate(dti_list_sorted):
        if not t1_list:
            continue
        t1_stem = t1_list[i] if i < len(t1_list) else t1_list[-1]
        dti_to_t1[dti_stem] = t1_stem
print(f"Matched {len(dti_to_t1)} DTI scans to T1 scans")

Matched 801 DTI scans to T1 scans


In [34]:
def _rigid_md_to_t1(md_path):
    dti_stem = md_path.name.replace("_MD.nii.gz", "")
    t1_stem = dti_to_t1.get(dti_stem)
    fa_path = md_path.parent / f"{dti_stem}_FA.nii.gz"
    
    if t1_stem is None:
        return dti_stem, "fail", "no matching T1 for subject"

    out_md = md_t1_dir / f"{dti_stem}_MD.nii.gz"
    out_tx = md_t1_dir / f"{dti_stem}_rigid.mat"

    if out_md.exists() and out_tx.exists():
        return dti_stem, "skip", ""

    t1_path = t1_strip_dir / f"{t1_stem}.nii.gz"
    if not t1_path.exists():
        return dti_stem, "fail", f"missing T1: {t1_path.name}"

    try:
        fixed = ants.image_read(str(t1_path))
        moving_fa = ants.image_read(str(fa_path))   # register on FA, not MD
        reg = ants.registration(fixed=fixed, moving=moving_fa, type_of_transform="Rigid")
        # Save FA in T1 space (sanity-check map)
        ants.image_write(reg["warpedmovout"], str(md_t1_dir / f"{dti_stem}_FA.nii.gz"))
        # Save the rigid .mat
        for tx in reg["fwdtransforms"]:
            if Path(tx).suffix == ".mat":
                shutil.copy(tx, md_t1_dir / f"{dti_stem}_rigid.mat")
        # Apply the same transform to MD
        md_img = ants.image_read(str(md_path))
        md_in_t1 = ants.apply_transforms(
            fixed=fixed, moving=md_img,
            transformlist=reg["fwdtransforms"],
            interpolator="linear",
        )
        ants.image_write(md_in_t1, str(md_t1_dir / f"{dti_stem}_MD.nii.gz"))

        for tx_path in reg["fwdtransforms"]:
            if Path(tx_path).suffix == ".mat" or "GenericAffine" in Path(tx_path).name:
                shutil.copy(tx_path, out_tx)
                break
        return dti_stem, "ok", ""
    except Exception as e:
        return dti_stem, "fail", str(e)

In [35]:
failed = []
skipped = 0
ok = 0

max_workers = max(1, (os.cpu_count() or 4) - 2)
print(f"Running rigid MD -> native T1 with {max_workers} workers")

with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = [ex.submit(_rigid_md_to_t1, p) for p in md_maps]
    for fut in tqdm(as_completed(futures), total=len(md_maps), desc="Rigid MD -> T1"):
        stem, status, err = fut.result()
        if status == "ok":
            ok += 1
        elif status == "skip":
            skipped += 1
        else:
            failed.append((stem, err))

reg_files = list(md_t1_dir.glob("*_MD.nii.gz"))
print(f"\nRigid MD -> native T1 complete:")
print(f"  {len(reg_files)} MD maps in native T1 space")
print(f"  ({ok} new, {skipped} skipped, {len(failed)} failed)")
if failed:
    for name, err in failed[:10]:
        print(f"  {name}: {err}")

Running rigid MD -> native T1 with 14 workers


Rigid MD -> T1:  15%|█▍        | 118/801 [04:49<24:40,  2.17s/it]Exception Object caught: 

itk::ExceptionObject (0x16fdd5b30)
Location: "unknown" 
File: /Users/runner/work/ANTsPy/ANTsPy/itksource/Modules/Filtering/ImageStatistics/include/itkImageMomentsCalculator.hxx
Line: 124
Description: ITK ERROR: ImageMomentsCalculator(0x1228a0fb0): Compute(): Total Mass of the image was zero. Aborting here to prevent division by zero later on.


Rigid MD -> T1:  26%|██▋       | 211/801 [09:05<21:12,  2.16s/it]Exception Object caught: 

itk::ExceptionObject (0x1228dd200)
Location: "unknown" 
File: /Users/runner/work/ANTsPy/ANTsPy/itksource/Modules/Filtering/ImageStatistics/include/itkImageMomentsCalculator.hxx
Line: 124
Description: ITK ERROR: ImageMomentsCalculator(0x1337bd2a0): Compute(): Total Mass of the image was zero. Aborting here to prevent division by zero later on.


Rigid MD -> T1:  29%|██▉       | 231/801 [10:05<30:38,  3.23s/it]Exception Object caught: 

itk::ExceptionObject (0x326b4c


Rigid MD -> native T1 complete:
  787 MD maps in native T1 space
  (787 new, 0 skipped, 14 failed)
  I1092244_114_S_6039: Registration failed with error code 1
  I1224877_020_S_6513: Registration failed with error code 1
  I1241880_020_S_5203: Registration failed with error code 1
  I1278648_020_S_6185: Registration failed with error code 1
  I1325865_020_S_6449: Registration failed with error code 1
  I1329976_020_S_6513: Registration failed with error code 1
  I1332415_020_S_6227: Registration failed with error code 1
  I1349126_020_S_6504: Registration failed with error code 1
  I1412031_020_S_6901: Registration failed with error code 1
  I1428968_022_S_5004: Registration failed with error code 1


In [36]:
from concurrent.futures import ThreadPoolExecutor, as_completed

t1_warps_dir = Path("model_data/adni/t1_long_data/t1_long_modulated_vbm/t1_long_vbm_warps")
mni_t1_path = Path("model_data/mni_t1.nii")
mni_t1 = ants.image_read(str(mni_t1_path))

dti_vbm_root = Path("model_data/adni/dti_long_data/dti_long_modulated_vbm")
md_t1_dir = dti_vbm_root / "dti_long_reg_t1_native"
md_mni_dir = dti_vbm_root / "dti_long_mni"

md_t1_files = sorted(md_t1_dir.glob("*_MD.nii.gz"))
print(f"Found {len(md_t1_files)} MD maps in native T1 space to warp into MNI")

Found 787 MD maps in native T1 space to warp into MNI


In [37]:
def _warp_md_to_mni(md_t1_path):
    dti_stem = md_t1_path.name.replace("_MD.nii.gz", "")
    t1_stem = dti_to_t1.get(dti_stem)
    if t1_stem is None:
        return dti_stem, "fail", "no T1 mapping"

    out_md = md_mni_dir / f"{dti_stem}_MD.nii.gz"
    if out_md.exists():
        return dti_stem, "skip", ""

    affine = t1_warps_dir / f"{t1_stem}_0GenericAffine.mat"
    warp = t1_warps_dir / f"{t1_stem}_1Warp.nii.gz"
    missing = [p for p in [affine, warp] if not p.exists()]
    if missing:
        return dti_stem, "fail", f"missing warps: {[p.name for p in missing]}"

    try:
        moving = ants.image_read(str(md_t1_path))
        warped = ants.apply_transforms(
            fixed=mni_t1,
            moving=moving,
            transformlist=[str(warp), str(affine)],
            interpolator="linear",
        )
        ants.image_write(warped, str(out_md))
        return dti_stem, "ok", ""
    except Exception as e:
        return dti_stem, "fail", str(e)

In [38]:
failed = []
skipped = 0
ok = 0

max_workers = max(1, (os.cpu_count() or 4) - 2)
print(f"Running apply_transforms MD -> MNI with {max_workers} workers")

with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = [ex.submit(_warp_md_to_mni, p) for p in md_t1_files]
    for fut in tqdm(as_completed(futures), total=len(md_t1_files), desc="Warp MD -> MNI"):
        stem, status, err = fut.result()
        if status == "ok":
            ok += 1
        elif status == "skip":
            skipped += 1
        else:
            failed.append((stem, err))

mni_files = list(md_mni_dir.glob("*_MD.nii.gz"))
print(f"\nWarp MD -> MNI complete:")
print(f"  {len(mni_files)} MD maps in MNI space")
print(f"  ({ok} new, {skipped} skipped, {len(failed)} failed)")
if failed:
    for name, err in failed[:10]:
        print(f"  {name}: {err}")

Running apply_transforms MD -> MNI with 14 workers


Warp MD -> MNI: 100%|██████████| 787/787 [02:59<00:00,  4.39it/s]


Warp MD -> MNI complete:
  787 MD maps in MNI space
  (787 new, 0 skipped, 0 failed)


In [39]:
from concurrent.futures import ThreadPoolExecutor, as_completed

dti_vbm_root = Path("model_data/adni/dti_long_data/dti_long_modulated_vbm")
md_mni_dir = dti_vbm_root / "dti_long_mni"
md_smooth_dir = dti_vbm_root / "dti_long_mni_smoothed"

fwhm_mm = 5.0
sigma_mm = fwhm_mm / (2.0 * np.sqrt(2.0 * np.log(2.0)))
print(f"Smoothing MD at {fwhm_mm} mm FWHM (sigma = {sigma_mm:.4f} mm)")

mni_files = sorted(md_mni_dir.glob("*_MD.nii.gz"))
print(f"Found {len(mni_files)} MD maps in MNI to smooth")

Smoothing MD at 5.0 mm FWHM (sigma = 2.1233 mm)
Found 787 MD maps in MNI to smooth


In [40]:
def _smooth_md(md_mni_path):
    dti_stem = md_mni_path.name.replace("_MD.nii.gz", "")
    out_path = md_smooth_dir / f"{dti_stem}_MD_s5.nii.gz"
    if out_path.exists():
        return dti_stem, "skip", ""
    try:
        img = ants.image_read(str(md_mni_path))
        smoothed = ants.smooth_image(
            img,
            sigma=sigma_mm,
            sigma_in_physical_coordinates=True,
        )
        ants.image_write(smoothed, str(out_path))
        return dti_stem, "ok", ""
    except Exception as e:
        return dti_stem, "fail", str(e)

In [41]:
failed = []
skipped = 0
ok = 0

max_workers = max(1, (os.cpu_count() or 4) - 2)
print(f"Smoothing with {max_workers} workers")

with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = [ex.submit(_smooth_md, p) for p in mni_files]
    for fut in tqdm(as_completed(futures), total=len(mni_files), desc="Smooth MD"):
        stem, status, err = fut.result()
        if status == "ok":
            ok += 1
        elif status == "skip":
            skipped += 1
        else:
            failed.append((stem, err))

smoothed_files = list(md_smooth_dir.glob("*_MD_s5.nii.gz"))
print(f"\nMD smoothing complete:")
print(f"  {len(smoothed_files)} smoothed MD maps")
print(f"  ({ok} new, {skipped} skipped, {len(failed)} failed)")
if failed:
    for name, err in failed[:10]:
        print(f"  {name}: {err}")

Smoothing with 14 workers


Smooth MD: 100%|██████████| 787/787 [00:57<00:00, 13.77it/s]


MD smoothing complete:
  787 smoothed MD maps
  (787 new, 0 skipped, 0 failed)


#### ML dataset pre processing

In [46]:
# --- Build GM / WM masks resampled to the smoothed-image grid ----------------
# MNI tissue masks are 1 mm; smoothed PVEs/MD are 1.5 mm -> resample (NN).
import nilearn.image as nli

t1_smooth_dir = Path("model_data/adni/t1_long_data/t1_long_modulated_vbm/t1_long_pve_mni_smoothed")
dti_smooth_dir = Path("model_data/adni/dti_long_data/dti_long_modulated_vbm/dti_long_mni_smoothed")
out_t1_dir = Path("model_data/adni/t1_long_data")
out_dti_dir = Path("model_data/adni/dti_long_data")
meta_csv = Path("model_data/adni/paired_df_long.csv")

ref_img = nib.load(sorted(dti_smooth_dir.glob("*_MD_s5.nii.gz"))[0])
ref_shape = ref_img.shape

def _load_mask(path):
    m = nli.resample_to_img(str(path), ref_img, interpolation="nearest", force_resample=True, copy_header=True)
    return np.asarray(m.dataobj) > 0.5

gm_mask = _load_mask("model_data/mni_gm_mask.nii")
wm_mask = _load_mask("model_data/mni_wm_mask.nii")
print(f"Reference grid: {ref_shape}  |  GM voxels: {gm_mask.sum():,}  |  WM voxels: {wm_mask.sum():,}")

Reference grid: (121, 145, 121)  |  GM voxels: 451,681  |  WM voxels: 283,320


In [57]:
# --- Pair T1 <-> DTI by subject_id + scan order, then build T1 parquets ------
# Stem = "I<image_id>_<subject_id>" (e.g. I1001084_003_S_6260)
from concurrent.futures import ThreadPoolExecutor

def _parse(stem):
    img, _, subj = stem.partition("_")
    return img.lstrip("I"), subj

t1_subjects = sorted([p for p in t1_smooth_dir.iterdir() if p.is_dir()])
dti_files = sorted(dti_smooth_dir.glob("*_MD_s5.nii.gz"))
t1_all = pd.DataFrame([(p.name, *_parse(p.name)) for p in t1_subjects],
                      columns=["t1_image_subject_id", "t1_image_id", "subject_id"])
dti_all = pd.DataFrame([(s := f.name.replace("_MD_s5.nii.gz", ""), *_parse(s)) for f in dti_files],
                       columns=["dti_image_subject_id", "dti_image_id", "subject_id"])
t1_all = t1_all.sort_values(["subject_id", "t1_image_id"]).reset_index(drop=True)
dti_all = dti_all.sort_values(["subject_id", "dti_image_id"]).reset_index(drop=True)
t1_all["scan_n"] = t1_all.groupby("subject_id").cumcount()
dti_all["scan_n"] = dti_all.groupby("subject_id").cumcount()
paired_stems = t1_all.merge(dti_all, on=["subject_id", "scan_n"], how="inner").drop(columns="scan_n")
print(f"T1: {len(t1_all)}  DTI: {len(dti_all)}  Paired: {len(paired_stems)}")

paired_t1_dirs = [t1_smooth_dir / s for s in paired_stems["t1_image_subject_id"]]

def _extract_t1(subj_dir):
    stem = subj_dir.name
    gm = nib.load(subj_dir / f"{stem}_pve_1_mod_s2p5.nii.gz").get_fdata(dtype=np.float32)
    wm = nib.load(subj_dir / f"{stem}_pve_2_mod_s2p5.nii.gz").get_fdata(dtype=np.float32)
    return stem, gm[gm_mask], wm[wm_mask]

stems, gm_rows, wm_rows = [], [], []
with ThreadPoolExecutor(max_workers=max(1, (os.cpu_count() or 4) - 2)) as ex:
    for stem, gm_vec, wm_vec in tqdm(ex.map(_extract_t1, paired_t1_dirs), total=len(paired_t1_dirs), desc="T1 mask -> parquet"):
        stems.append(stem); gm_rows.append(gm_vec); wm_rows.append(wm_vec)

gm_df = pd.DataFrame(np.vstack(gm_rows), index=pd.Index(stems, name="image_subject_id"))
wm_df = pd.DataFrame(np.vstack(wm_rows), index=pd.Index(stems, name="image_subject_id"))
gm_df.columns = gm_df.columns.astype(str); wm_df.columns = wm_df.columns.astype(str)
gm_df.to_parquet(out_t1_dir / "t1_long_masked_gm.parquet", compression="zstd")
wm_df.to_parquet(out_t1_dir / "t1_long_masked_wm.parquet", compression="zstd")
print(f"Saved t1_long_masked_gm.parquet {gm_df.shape} and t1_long_masked_wm.parquet {wm_df.shape}")

T1: 802  DTI: 787  Paired: 787


T1 mask -> parquet: 100%|██████████| 787/787 [00:09<00:00, 81.40it/s] 


Saved t1_long_masked_gm.parquet (787, 451681) and t1_long_masked_wm.parquet (787, 283320)


In [48]:
# --- DTI MD -> masked GM-MD / WM-MD parquets (paired set only) ---------------
def _extract_md(md_path):
    stem = md_path.name.replace("_MD_s5.nii.gz", "")
    md = nib.load(md_path).get_fdata(dtype=np.float32)
    return stem, md[gm_mask], md[wm_mask]

paired_dti_files = [dti_smooth_dir / f"{s}_MD_s5.nii.gz" for s in paired_stems["dti_image_subject_id"]]
print(f"Building DTI parquets for {len(paired_dti_files)} paired subjects")

stems, gm_rows, wm_rows = [], [], []
with ThreadPoolExecutor(max_workers=max(1, (os.cpu_count() or 4) - 2)) as ex:
    for stem, gm_vec, wm_vec in tqdm(ex.map(_extract_md, paired_dti_files), total=len(paired_dti_files), desc="DTI mask -> parquet"):
        stems.append(stem); gm_rows.append(gm_vec); wm_rows.append(wm_vec)

dti_gm = pd.DataFrame(np.vstack(gm_rows), index=pd.Index(stems, name="image_subject_id"))
dti_wm = pd.DataFrame(np.vstack(wm_rows), index=pd.Index(stems, name="image_subject_id"))
dti_gm.columns = dti_gm.columns.astype(str); dti_wm.columns = dti_wm.columns.astype(str)
dti_gm.to_parquet(out_dti_dir / "dti_long_masked_gm_md.parquet", compression="zstd")
dti_wm.to_parquet(out_dti_dir / "dti_long_masked_wm_md.parquet", compression="zstd")
print(f"Saved dti_long_masked_gm_md.parquet {dti_gm.shape} and dti_long_masked_wm_md.parquet {dti_wm.shape}")

Building DTI parquets for 787 subjects


DTI mask -> parquet: 100%|██████████| 787/787 [00:06<00:00, 114.38it/s]


Saved dti_long_masked_gm_md.parquet (787, 451681) and dti_long_masked_wm_md.parquet (787, 283320)


In [56]:
# --- paired_df_long.csv : metadata linking T1 + DTI rows to ADNI labels ------
paired = paired_stems.copy()

# DX from ADNIMERGE: most recent non-null DX per PTID
adni = pd.read_csv("model_data/adni/ADNIMERGE_22Dec2025.csv", usecols=["PTID", "EXAMDATE", "DX"], low_memory=False)
adni = adni.dropna(subset=["DX"]).sort_values("EXAMDATE").drop_duplicates("PTID", keep="last")
paired["group"] = paired["subject_id"].map(dict(zip(adni["PTID"], adni["DX"])))
paired["diag_label"] = paired["group"].map({"CN": 0, "MCI": 1, "Dementia": 1})

# Amyloid status from UCBERKELEY: most recent non-null per PTID
amy = pd.read_csv("model_data/adni/All_Subjects_UCBERKELEY_AMY_6MM_08Feb2026.csv",
                  usecols=["PTID", "SCANDATE", "AMYLOID_STATUS"], low_memory=False)
amy = amy.dropna(subset=["AMYLOID_STATUS"]).sort_values("SCANDATE").drop_duplicates("PTID", keep="last")
paired["amyloid_label"] = paired["subject_id"].map(dict(zip(amy["PTID"], amy["AMYLOID_STATUS"]))).astype("Int64")

paired = paired[["subject_id",
                 "t1_image_id", "dti_image_id",
                 "t1_image_subject_id", "dti_image_subject_id",
                 "group", "diag_label", "amyloid_label"]]
paired.to_csv(meta_csv, index=False)
print(f"Saved {meta_csv}  ({len(paired)} rows)")
print(paired["group"].value_counts(dropna=False).to_dict(),
      "| amyloid:", paired["amyloid_label"].value_counts(dropna=False).to_dict())

Saved model_data/adni/paired_df_long.csv  (787 rows)
{'CN': 383, 'MCI': 248, 'Dementia': 111, nan: 45} | amyloid: {np.int64(1): 376, np.int64(0): 348, <NA>: 63}
